<a href="https://colab.research.google.com/github/SaintJeane/Complete-Data-Science-With-Machine-Learning-And-NLP-2024/blob/main/agentic_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --quiet --upgrade langchain-community langchain-huggingface crawl4ai google-api-python-client gradio bitsandbytes accelerate
!pip install -qU langgraph
!crawl4ai-setup

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.2/426.2 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 118.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.0/325.0 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.2/291.2 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.

In [2]:
import logging
logging.basicConfig(filename="rag.log",
                    level=logging.DEBUG,
                    format="%(asctime)s - %(levelname)s - %(message)s",
                    datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger(__name__)

In [3]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


In [4]:
import torch
import uuid
import re
import numpy as np
import scipy.stats as stats # For entropy calculation
from time import time
from datetime import datetime, timezone, timedelta
from typing import TypedDict, Annotated

import asyncio
import nest_asyncio
nest_asyncio.apply()

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from sentence_transformers import SentenceTransformer

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, get_buffer_string, ToolCall
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_core.messages import trim_messages, SystemMessage, ToolMessage

from langchain.agents.output_parsers import ReActSingleInputOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain_community.chat_message_histories import ChatMessageHistory

from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, END

from googleapiclient.discovery import build

from crawl4ai import AsyncWebCrawler, CrawlerRunConfig
from crawl4ai.markdown_generation_strategy import DefaultMarkdownGenerator
from crawl4ai.content_filter_strategy import PruningContentFilter

In [5]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [6]:
# Verify Hugging Face login in the notebook
from huggingface_hub import login
from google.colab import userdata

login(userdata.get('HF_TOKEN'))

In [7]:
# Setup the quantized Hugging Face model
model_id = "google/gemma-2-2b-it"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # try changing to torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quantization_config,
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

# Create a text generation pipeline
text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    max_new_tokens=256, # Try reducing to 256 for memory stability
    do_sample=True,
    temperature=0.3, # test with 0.5
)

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Device set to use cuda:0


In [8]:
## Utilities
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module='huggingface_module')

# Embeddings + FAISS
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
# embedding_dim = embedding_model.get_sentence_embedding_dimension()

# Colab notebook setup
from google.colab import userdata
GOOGLE_API_KEY = userdata.get("API_GOOGLE_SEARCH")
SEARCH_ENGINE_ID = userdata.get("SEARCH_ENGINE_ID")

vectorstore = None # will hold FAISS index
seen_urls = {} # Track URLs already crawled
EXPIRY_DAYS = 1 # Keep data for 24 hrs

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
import torch
import gc

def clean_gpu():
  """A simple function for releasing the GPU memory cache"""
  gc.collect()
  if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

clean_gpu()

In [10]:
def chunk_and_index(text: str, source_url: str = None):
  """Split text into chunks and upsert into FAISS with overwrite per URL + timestamp."""
  global vectorstore, seen_urls
  splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
  chunks = splitter.split_text(text)

  # LangChain expects list of Documents
  docs = [Document(page_content=chunk, metadata={"source": source_url or "unknown",})
          for chunk in chunks]
  ids = [str(uuid.uuid4()) for _ in docs]

  if vectorstore is None:
    vectorstore = FAISS.from_documents(docs, embedding_model, ids=ids)
    vectorstore.save_local("faiss_index")  # persist
  else:
    # If URL already exists → delete old entries then add
    if source_url in seen_urls:
        vectorstore.delete(ids=seen_urls[source_url]["ids"])

    # Add new docs
    vectorstore.add_documents(docs, ids=ids)
    vectorstore.save_local("faiss_index")

  # Update seen_urls
  seen_urls[source_url] = {
        "ids": ids,
        "last_crawled": datetime.now(timezone.utc)
    }

  del docs, chunks
  gc.collect
  clean_gpu()

In [11]:
## subclass `HuggingFacePipeline` so that `.invoke()` extracts just the "input" field

class SafeHFPipeline(HuggingFacePipeline):
  def _call(self, prompt:str, stop=None):
    # If dict, grab "input"
    if isinstance(prompt, dict) and "input" in prompt:
      prompt = prompt["input"]
    response = super()._call(prompt, stop=stop)
    print(f"DEBUG: RAW LLM RESPONSE:\n---\n{response}\n---")
    return response

hf_pipeline = SafeHFPipeline(pipeline=text_generation_pipeline)

In [12]:
def print_gpu_mem(note=""):
  """Helper function to help monitor GPU usage"""
  if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved() / 1e9
    print(f"[GPU] {note} | Allocated: {allocated:.2f} GB | Reserved: {reserved:.2f} GB")


In [13]:
def count_tokens(texts, tokenizer: AutoTokenizer):
  """Ensures only list of strings are passed to the Hugging Face tokenizer"""
  if isinstance(texts, list):
      # Extract content from LangChain message objects
      texts = [m.content if hasattr(m, 'content') else str(m) for m in texts]
  elif not isinstance(texts, str):
      texts = [str(texts)] # Fallback

  # Ensure it's a list of strings for the tokenizer
  if isinstance(texts, str):
      texts = [texts]

  # Use the tokenizer to get the token count
  encodings = tokenizer(texts, padding=False, truncation=False)
  return sum(len(ids) for ids in encodings["input_ids"])

In [14]:
class AgentState(TypedDict):
  """
  The state of the agent's workflow.
  Storing and tracking structured info.
  """
  messages: Annotated[list[BaseMessage], add_messages]
  query: str
  retrieved_docs: list[dict] | None
  search_results: list[dict] | None
  answer: str | None

In [15]:
# Helper function for truncating messages
MAX_TURNS = 3 # Keep last 3 exchanges (user + assistant)
MODEL_MAX_TOKENS = 8192
SAFETY_MARGIN = 256

## Remove (redundant)
# def truncate_messages(messages):
#   """
#   Truncate chat history to last MAX_TURNS exchanges.
#   """
#   # messages is a list of (role, content) or ChatMessages
#   # Keep system prompt if present
#   system_msgs = [m for m in messages if m.type == "system"]
#   dialog_msgs = [m for m in messages if m.type != "system"]

#   # Keep only the last MAX_TURNS * 2 (user+AI) messages
#   truncated_dialog = dialog_msgs[-(MAX_TURNS * 2):]

#   return system_msgs + truncated_dialog

In [16]:
# Adaptive k for dynamic retrieval depth
def adaptive_k(query: str) -> int:
  """This adjusts how many documents to retrieve based on query complexity."""
  length = len(query.split())
  if length > 20:
      return 12  # Complex query
  elif length > 10:
      return 8   # Moderate query
  else:
      return 6   # Simple query

# Define combined pruning function
def prune_chat_history(history, max_turns=3, max_tokens=500):
  """
   Trim chat history by keeping the last `max_turns` messages,
  then applying a token-based trim to stay under `max_tokens`.
  Preserves correct LangChain message types.
  Includes debug logging for visibility.
  """

  logger.info("=== [PRUNE] Starting chat history pruning ===")
  logger.info(f"Initial message count: {len(history.messages)}")

  # Step 1: Rolling window – keep last `max_turns` messages
  history.messages = history.messages[-max_turns:]
  logger.info(f"After rolling window (max_turns={max_turns}): {len(history.messages)} messages kept")

  # Step 2: Force all messages into plain strings fpr safe token counting
  message_texts = [get_buffer_string([m]) for m in history.messages]
  # Ensure message_texts is a list of strings
  if not all(isinstance(text, str) for text in message_texts):
      message_texts = [str(text) for text in message_texts]

  logging.debug(f"Message texts (pre-trim): {message_texts}")

  # Step 3: Token-based pruning – trim to `max_tokens`
  # Trim by tokens: only messages as positional arg, others keyword
  trimmed_messages = trim_messages(
      messages=history.messages, # Pass message objects directly
      max_tokens=max_tokens,
      token_counter=lambda texts: count_tokens(texts, tokenizer),
      strategy="last",
      include_system=True,
      start_on="human",
      end_on=("human", "tool"),
    )
  logger.info(f"After token trimming (max_tokens={max_tokens}): {len(trimmed_messages)} messages remain")
  logger.debug(f"Message texts (post-trim): {trimmed_messages}")

  # Step 4: rebuild messages with original types
  rebuilt = []
  for old_msg, trimmed_msg in zip(history.messages, trimmed_messages):
      # Extract content from the trimmed message object
      new_text = trimmed_msg.content
      if isinstance(old_msg, HumanMessage):
          rebuilt.append(HumanMessage(content=new_text))
      elif isinstance(old_msg, AIMessage):
          rebuilt.append(AIMessage(content=new_text))
      elif isinstance(old_msg, SystemMessage):
          rebuilt.append(SystemMessage(content=new_text))
      elif isinstance(old_msg, ToolMessage):
          rebuilt.append(ToolMessage(content=new_text, tool_call_id=getattr(old_msg, "tool_call_id", None)))
      else:
          logger.warning(f"Unknown message type {type(old_msg)}, falling back to HumanMessage")
          rebuilt.append(HumanMessage(content=new_text))

  history.messages = rebuilt
  logger.info(f"=== [PRUNE] Done. Final message count: {len(history.messages)} ===")

  return history

# Pruning action for FAISS to overwrite same URLs to prevent duplication
def prune_vectorstore(vectorstore):
  """Delete docs older than EXPIRY_DAYS using seen_urls metadata"""
  if vectorstore is None:
    return

  now = datetime.now(timezone.utc)
  cutoff = timedelta(days=EXPIRY_DAYS)

  # Collect IDs of expired docs
  to_delete = []
  for url, entry in list(seen_urls.items()):
    if now - entry["last_crawled"] > cutoff:
      to_delete.extend(entry["ids"])
      del seen_urls[url]

  if to_delete:
      vectorstore.delete(to_delete)

In [19]:
## Tools
# 1. Internal retriever (with uncertainty scoring)
@tool
def internal_retriever(query: str):
  """
  Search internal FAISS store for relevant docs with uncertainty scoring (LLM readable).
  Return dicts for state and formatted strings for LLM readability.
  """
  global vectorstore
  if vectorstore is None:
    return {
            "results": [],
            "top1_score": 0.0,
            "entropy": 1.0,
            "llm_text": "No documents indexed yet. top1_score=0.0 entropy=1.0"
           }
  # k = adaptive_k(query) # **Adaptive retrieval depth
  k = 6,
  # retriever = vectorstore.as_retriever(search_kwargs={"k": k})
  # docs = retriever.get_relevant_documents(query)
  # Use built-in FAISS functionality for efficient similarity search and scoring
  docs_with_scores = vectorstore.similarity_search_with_score(query, k=k)
  if not docs_with_scores:
    return {
            "results": [],
            "top1_score": 0.0,
            "entropy": 1.0,
            "llm_text": "No documents found. top1_score=0.0 entropy=1.0"
           }
  # # Compute similarity scores manually (via embeddings)
  # query_vec = embedding_model.encode(query, convert_to_numpy=True)
  # doc_texts = [d.page_content for d in docs]
  # # doc_vecs = [embedding_model.embed_query(d.page_content) for d in docs]
  # doc_vecs = embedding_model.encode(doc_texts, convert_to_numpy=True, batch_size=32, show_progress_bar=False)

  # # Cosine similarity
  # sims = np.dot(doc_vecs, query_vec) / (np.linalg.norm(doc_vecs, axis=1) * (np.linalg.norm(query_vec) + 1e-10))

  docs = [d for d, _ in docs_with_scores]
  sims = np.array([score for _, score in docs_with_scores])

  # Normalize for entropy
  sims = np.maximum(sims, 1e-10)
  probs = sims / (sims.sum() + 1e-10)

  entropy = float(stats.entropy(probs, base=2))
  top1_score = float(np.max(sims))

  # Format output to be LLM readable
  results_text = "\n".join(
        [f"- (score={float(score):.3f}) {d.page_content[:200]}..." for d, score in zip(docs, sims)]
    )
  return {
        "results": docs,
        "top1_score": top1_score,
        "entropy": entropy,
        "llm_text": f"""
                        Retrieved {len(docs)} documents.
                        top1_score={top1_score:.3f}
                        entropy={entropy:.3f}
                        Docs:
                        {results_text}
                    """
    }
# 2. Google Custom Search
@tool
def google_search(query: str, num: int = 6):
  """Search Google Custom Search API for relevant pages."""
  service = build("customsearch", "v1", developerKey=GOOGLE_API_KEY)
  try:
      res = service.cse().list(q=query, cx=SEARCH_ENGINE_ID, num=num).execute()
      items = res.get("items", [])
      if not items:
          return "No results found from Google Search."

      # Format the list of dictionaries into a single string
      formatted_results = []
      for item in items:
          result = (
              f"Title: {item.get('title', 'N/A')}\n"
              f"URL: {item.get('link', 'N/A')}\n"
              f"Snippet: {item.get('snippet', 'N/A')}\n"
          )
          formatted_results.append(result)

      return "\n---\n".join(formatted_results)

  except Exception as e:
      return f"An error occurred during Google Search: {e}"

# 3. Crawl4AI scraping
@tool
def crawl4ai_fetch(url: str) -> str:
  """Fetch and clean page content via Crawl4AI and index into FAISS, overwrites FAISS entry if URL already exists."""
  run_config = CrawlerRunConfig(
    markdown_generator=DefaultMarkdownGenerator(
      content_filter=PruningContentFilter(threshold=0.5)
    )
  )
  async def fetch():
    async with AsyncWebCrawler() as crawler:
      result = await crawler.arun(url=url, config=run_config)
      if result.success and result.markdown:
        text = result.markdown.raw_markdown

        # Check for excessive text length before chunking
        if len(text) > 10_000_000: # Example limit, adjust as needed
          return f"Page {url} is too large to process and was skipped."

        # Index into FAISS immediately
        chunk_and_index(text, source_url=url)
        # auto-prune daily
        prune_vectorstore(vectorstore)
        last = seen_urls[url]["last_crawled"].isoformat()

        # Truncate output to prevent excessively long tool messages
        output = f"Indexed/Updated: {url} (last_crawled={last}). A large document was processed and indexed."
        return output[:MODEL_MAX_TOKENS - SAFETY_MARGIN] # Example truncation
        # return f"Indexed/Updated: {url} (last_crawled={last})"
        # return text
      return f"Failed to fetch {url}"
  return asyncio.run(fetch())

In [22]:
def parse_react_output(text: str) -> AIMessage:
  # Look for Action and Action Input
  action_match = re.search(r"Action:\s*(\w+)", text)
  input_match = re.search(r"Action Input:\s*(.*)", text)

  if action_match:
    tool_name = action_match.group(1).strip()
    tool_input = input_match.group(1).strip() if input_match else ""
    # Generate a unique ID for the tool call
    tool_call_id = str(uuid.uuid4())

    return AIMessage(
      content=text,  # keep raw text for debugging
      tool_calls=[ToolCall(name=tool_name, args={"query": tool_input}, id=tool_call_id)] # Add the unique ID
    )
  else:
    # No action found → Final Answer
    return AIMessage(content=text)

In [23]:
rag_agent_prompt = ChatPromptTemplate.from_template("""
system
You are a smart and efficient Research Agent. Your goal is to answer the user's question accurately, citing your sources.

You have access to the following tools:
{tools}

**Your Guiding Principles:**
- **Start Internally:** Always begin by using the `internal_retriever` tool. It's the fastest way to find an answer.
- **Evaluate and Decide:** After retrieving, your most important task is to *think* about the results. In your 'Thought' step, briefly assess the output. Are the results relevant (high `top1_score`) and clear (low `entropy`)?
- **Be Resourceful:** If the internal results are insufficient, ambiguous, out of date, or irrelevant, you must use `Google Search` to find better information from the web.
- **Be Selective:** When you search the web, don't crawl every link. Analyze the titles and snippets to identify the **2-3 most promising sources** and use `crawl4ai_fetch` on only those.
- **Refresh Your Knowledge:** After fetching new information, you **must** use `internal_retriever` one last time to query all the combined knowledge. This is a critical step.
- If you do not call at least one tool before your Final Answer, your response is invalid.

**Output Format:**
When you need to use a tool:
Thought: [Your reasoning based on the principles above. e.g., "The internal results have a low score, so I need to search the web."]
Action: [one of: {tool_names}]
Action Input: [the tool input]
Observation: [the result of the action]

When you have the final answer:
Thought: I have gathered and synthesized enough information to provide a conclusive answer.
Final Answer: [Your final, comprehensive answer with respsective urls as citations.]

Begin!

system
Previous conversation history:
{chat_history}

human
New input: {input}
{agent_scratchpad}
""")

In [24]:
def prune_node(state:AgentState) -> AgentState:
  """Wrapper node that plugs into the workflow"""
  # Create a ChatMessageHistory object from the list of messages
  chat_history_obj = ChatMessageHistory(messages=state["messages"])

  # Robust token-based pruning
  target_max_tokens = MODEL_MAX_TOKENS - SAFETY_MARGIN

  # Pass the ChatMessageHistory object to prune_chat_history
  state["messages"] = prune_chat_history(chat_history_obj,
                                         max_turns=MAX_TURNS,
                                         max_tokens=target_max_tokens).messages
  return state

In [25]:
tools = [internal_retriever, google_search, crawl4ai_fetch]
tool_node = ToolNode(tools)

In [26]:
# def call_model(state):
#     """
#     Invokes the LLM with rag_agent_prompt and returns an AIMessage.
#     Prints GPU usage and token count to debug OOM issues.
#     """
#     # Build inputs for the chain
#     inputs = {
#         "input": state["messages"][-1].content,  # Latest user query
#         "chat_history": state["messages"],       # full history for context
#         "agent_scratchpad": "",                  # required by template
#         "tool_names": [t.name for t in tools],   # Pass tool names
#         "tools": tools                           # Pass the tools list
#     }

#     # Render the full prompt text
#     prompt_text = rag_agent_prompt.format(**inputs)

#     # Count tokens with Gemma’s tokenizer
#     token_count = len(tokenizer(prompt_text)["input_ids"])
#     print(f"[DEBUG] Prompt length = {token_count} tokens")

#     # Show GPU before running
#     print_gpu_mem("Before generation")

#     # Run the model (pipeline)
#     raw_response = hf_pipeline(prompt_text)

#     # Show GPU after running
#     print_gpu_mem("After generation")

#     # Handle output
#     text = raw_response[0]["generated_text"] if isinstance(raw_response, list) else str(raw_response)
#     parsed = parse_react_output(text)

#     return {"messages": [parsed]}

In [27]:
MAX_PROMPT_TOKENS = 4000 # Adjust

def call_model(state: AgentState) -> AgentState:
  """
  Invokes the LLM with rag_agent_prompt and returns an AIMessage.
  """
  # Truncate history before sending
  # messages = truncate_messages(state["messages"])
  # state["messages"] = truncate_messages(state["messages"])
  messages = state["messages"]

  # Build prompt text
  prompt_text = rag_agent_prompt.format(
      input=messages[-1].content,         # latest user query
      chat_history=messages,              # truncated history
      agent_scratchpad="",
      tool_names=[t.name for t in tools],
      tools=tools
  )

  # Debug token length
  enc = tokenizer(prompt_text, return_tensors="pt").input_ids
  token_len = enc.size(1)
  print(f"[DEBUG] Prompt length = {token_len} tokens")

  # Hard cut-off if too long
  if token_len > MAX_PROMPT_TOKENS:
    print(f"[WARN] Prompt too long ({token_len}), truncating to {MAX_PROMPT_TOKENS}")
    # Trim tokens down
    enc = enc[:, -MAX_PROMPT_TOKENS:]
    prompt_text = tokenizer.decode(enc[0], skip_special_tokens=True)

  # Show GPU before running
  print_gpu_mem("Before generation")

  # Run the model
  raw_response = hf_pipeline(prompt_text)

  # Show GPU after running
  print_gpu_mem("After generation")

  # Extract text
  text = raw_response[0]["generated_text"] if isinstance(raw_response, list) else str(raw_response)

  # Parse into AIMessage
  parsed = parse_react_output(text)

  # Clear transient state to prevent accumulation
  state["retrieved_docs"] = None
  state["search_results"] = None

  return {"messages": [parsed]}

In [28]:
# # parser = ReActSingleInputOutputParser() # to convert Mistral's text into structured tool calls

# def call_model(state):
#   """
#   Invokes the LLM with rag_agent_prompt and returns an AIMessage.
#   Pass full messages, not just last user string.
#   """
#   # The prompt and LLM are invoked here as a chain.
#   chain = rag_agent_prompt | hf_pipeline

#   # LangGraph handles the parsing of the LLM's output and determines if it's a tool call.
#   # We pass the messages from the state to the LLM.
#   raw_response = chain.invoke({"input": state["messages"][-1].content, # Latest user query
#                               "chat_history": state["messages"], # full history for context
#                               "agent_scratchpad": "", # required by template
#                               "tool_names": [t.name for t in tools], # Pass tool names
#                               "tools": tools # Pass the tools list
#                               })
#   # raw_response may be a dict or AIMessage
#   text = raw_response.content if hasattr(raw_response, "content") else str(raw_response)

#   # Convert ReAct text → AIMessage with tool_calls
#   parsed = parse_react_output(text)

#   return {"messages": [parsed]}

In [29]:
# def should_continue(state: AgentState):
#   """Function to determine which path to take after the LLM's response."""
#   last_message = state['messages'][-1]

#   # Some LLM wrappers return dicts instead of Message objects
#   if hasattr(last_message, "tool_calls") and last_message.tool_calls:
#     return "continue"
#   # Otherwise, it's a Final Answer, so we end the graph.
#   else:
#     return "end"

In [30]:
def should_continue(state: AgentState):
  """Decide whether to continue tool use or end the graph."""
  last_message = state['messages'][-1]

  # Handle dict-like or object-like message
  tool_calls = getattr(last_message, "tool_calls", None)
  if tool_calls and len(tool_calls) > 0:
    return "continue"
  # Otherwise, Final Answer
  return "end"

In [31]:
# Build the graph
workflow = StateGraph(AgentState)

# Add the nodes
workflow.add_node("prune", prune_node)
workflow.add_node("llm", call_model)
workflow.add_node("tools", tool_node)

# Set the entry point (the first node to execute, always prune first)
workflow.set_entry_point("prune")
# Ensure pruning happens before the LLM node
workflow.add_edge("prune", "llm")

# Add the conditional edges
workflow.add_conditional_edges(
  "llm",  # Start from the LLM node, passed as a positional argument
  should_continue,
  {
      "continue": "tools", # If condition is "continue", route to the "tools" node
      "end": END           # If condition is "end", finish the graph
  }
)

# After a tool is executed, always loop back to the LLM to let it reason on the result
# workflow.add_edge('tools', 'llm')
workflow.add_edge('tools', 'prune')
workflow.add_edge('prune', 'llm')

# Compile the graph
app = workflow.compile()

In [32]:
# Run Agent (Async)
async def run_agent(query: str, session_id: str='default'):
  # Invoke the compiled LangGraph app with the initial message
  inputs = {"messages": [("user", query)]}
  result = await app.ainvoke(inputs)

  # The final answer is the content of the last message in the state
  # return result['messages'][-1].content
  return result # Return the full state dictionary

In [33]:
import pprint as pprint

def ask():
  clean_gpu() # Call clean_gpu at the start of the loop
  question = input("Ask your question: ")
  full_state = asyncio.run(run_agent(question))
  raw_output = full_state['messages'][-1].content

  # Extract only the final section
  matches = re.findall(r"Final Answer:\s*(.*)", raw_output, re.DOTALL)
  if matches:
    final_answer = matches[-1].strip()
  else:
    final_answer = raw_output

  print("Answer:\n")
  print("--"*25)
  pprint.pprint(final_answer)

  clean_gpu()

ask()

Ask your question: Is Jennifer Lopez single?
[DEBUG] Prompt length = 785 tokens
[GPU] Before generation | Allocated: 2.22 GB | Reserved: 2.32 GB


/tmp/ipython-input-473418151.py:37: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  raw_response = hf_pipeline(prompt_text)


[GPU] After generation | Allocated: 2.23 GB | Reserved: 2.49 GB
[DEBUG] Prompt length = 5066 tokens
[WARN] Prompt too long (5066), truncating to 4000
[GPU] Before generation | Allocated: 2.23 GB | Reserved: 2.49 GB
[GPU] After generation | Allocated: 2.23 GB | Reserved: 4.26 GB
[DEBUG] Prompt length = 13254 tokens
[WARN] Prompt too long (13254), truncating to 4000
[GPU] Before generation | Allocated: 2.23 GB | Reserved: 4.26 GB
[GPU] After generation | Allocated: 2.23 GB | Reserved: 4.26 GB
[DEBUG] Prompt length = 20362 tokens
[WARN] Prompt too long (20362), truncating to 4000
[GPU] Before generation | Allocated: 2.23 GB | Reserved: 4.26 GB
[GPU] After generation | Allocated: 2.23 GB | Reserved: 4.26 GB
[DEBUG] Prompt length = 26967 tokens
[WARN] Prompt too long (26967), truncating to 4000
[GPU] Before generation | Allocated: 2.23 GB | Reserved: 4.26 GB
[GPU] After generation | Allocated: 2.23 GB | Reserved: 4.26 GB
[DEBUG] Prompt length = 34356 tokens
[WARN] Prompt too long (34356), t

In [ ]:
# import pprint as pprint

# def ask_loop():
#   print("Type 'quit' or 'exit' to exit the chat.\n")
#   while True:
#     clean_gpu() # Call clean_gpu at the start of the loop
#     question = input("Ask your question: ")
#     if question.lower() in ['quit', 'exit', 'q']:
#       print("Adios!👋")
#       break
#     full_state = asyncio.run(run_agent(question))
#     raw_output = full_state['messages'][-1].content

#     # Extract only the final section
#     matches = re.findall(r"Final Answer:\s*(.*)", raw_output, re.DOTALL)
#     if matches:
#       final_answer = matches[-1].strip()
#     else:
#       final_answer = raw_output

#     print("Answer:\n")
#     print("--"*25)
#     pprint.pprint(final_answer)

#     clean_gpu()

# ask_loop()

In [ ]:
# import re
# import pprint

# def extract_outputs(full_state):
#     """
#     Extract the final answer and structured reasoning trace.
#     """
#     raw_output = full_state['messages'][-1].content

#     # --- Extract Final Answer ---
#     match = re.search(r"Final Answer:\s*(.*)", raw_output, re.DOTALL)
#     final_answer = match.group(1).strip() if match else raw_output.strip()

#     # --- Extract Trace (everything before Final Answer) ---
#     trace_text = re.sub(r"Final Answer:.*", "", raw_output, flags=re.DOTALL).strip()

#     # --- Parse structured steps ---
#     steps = []
#     step_pattern = re.compile(
#         r"Thought:\s*(.*?)\s*Action:\s*(.*?)\s*Action Input:\s*(.*?)\s*Observation:\s*(.*?)(?=(Thought:|$))",
#         re.DOTALL
#     )

#     for m in step_pattern.finditer(trace_text):
#         step = {
#             "Thought": m.group(1).strip(),
#             "Action": m.group(2).strip(),
#             "Action Input": m.group(3).strip(),
#             "Observation": m.group(4).strip()
#         }
#         steps.append(step)

#     return final_answer, steps


# def ask_loop():
#     print("Type 'quit' or 'exit' to exit the chat.\n")
#     while True:
#         question = input("Ask your question: ")
#         if question.lower() in ['quit', 'exit', 'q']:
#             print("Adios!👋")
#             break

#         full_state = asyncio.run(run_agent(question))
#         final_answer, reasoning_steps = extract_outputs(full_state)

#         # --- User-facing answer ---
#         print("\nAnswer:\n")
#         print("--"*25)
#         print(final_answer)

#         # --- Debug / logging trace ---
#         print("\n[Debug Trace]\n")
#         print("--"*25)
#         pprint.pprint(reasoning_steps)

#         clean_gpu()